In [1]:
#import dependencies
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

#importing models
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

#preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

#metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


In [2]:
#importing data
file_path1 = r"C:\Users\lokan\Document\Introduction to Biological Systems\ENDSEM PROJECT\embeddings\protbert_features.csv"
file_path2 = r"C:\Users\lokan\Document\Introduction to Biological Systems\ENDSEM PROJECT\embeddings\bert_features.csv"
file_path3 = r"C:\Users\lokan\Document\Introduction to Biological Systems\ENDSEM PROJECT\embeddings\roberta_features.csv"


data1 = pd.read_csv(file_path1)
X1 = data1.iloc[:,:-2]
y1 = data1["Label"]

data2 = pd.read_csv(file_path2)
X2 = data2.iloc[:,:-2]
y2 = data2["Label"]

data3 = pd.read_csv(file_path3)
X3 = data3.iloc[:,:-2]
y3 = data3["Label"]


In [3]:
#Data preprocessing
X_train1,X_test1,y_train1,y_test1 = train_test_split(X1,y1,test_size=0.2, random_state=42)
X_train2,X_test2,y_train2,y_test2 = train_test_split(X2,y2,test_size=0.2, random_state=42)
X_train3,X_test3,y_train3,y_test3 = train_test_split(X3,y3,test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled1 = scaler.fit_transform(X_train1)
X_test_scaled1 = scaler.transform(X_test1)

X_train_scaled2 = scaler.fit_transform(X_train2)
X_test_scaled2 = scaler.transform(X_test2)

X_train_scaled3 = scaler.fit_transform(X_train3)
X_test_scaled3 = scaler.transform(X_test3)


In [4]:
def evaluate_model(model, X_train, X_test, y_train, y_test,
                   dataset_name, model_name, save_folder):

    # -----------------------------
    # TRAIN MODEL
    # -----------------------------
    model.fit(X_train, y_train)

    # -----------------------------
    # PREDICTIONS
    # -----------------------------
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # -----------------------------
    # METRICS
    # -----------------------------
    results = {
        "Train Accuracy": accuracy_score(y_train, y_train_pred),
        "Test Accuracy": accuracy_score(y_test, y_test_pred),

        "Train Precision": precision_score(
            y_train, y_train_pred,
            average='weighted',
            zero_division=0
        ),

        "Test Precision": precision_score(
            y_test, y_test_pred,
            average='weighted',
            zero_division=0
        ),

        "Train Recall": recall_score(
            y_train, y_train_pred,
            average='weighted',
            zero_division=0
        ),

        "Test Recall": recall_score(
            y_test, y_test_pred,
            average='weighted',
            zero_division=0
        ),

        "Train F1": f1_score(
            y_train, y_train_pred,
            average='weighted',
            zero_division=0
        ),

        "Test F1": f1_score(
            y_test, y_test_pred,
            average='weighted',
            zero_division=0
        )
    }    # =====================================================
    # CONFUSION MATRIX
    # =====================================================

    # Unique labels
    labels = np.unique(np.concatenate([y_train, y_test]))

    cm = confusion_matrix(y_test, y_test_pred, labels=labels)

    # -----------------------------
    # SAVE CONFUSION MATRIX VALUES
    # -----------------------------
    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual_{l}" for l in labels],
        columns=[f"Predicted_{l}" for l in labels]
    )

    csv_path = os.path.join(
        save_folder,
        f"{dataset_name}_{model_name}_confusion_matrix.csv"
    )

    cm_df.to_csv(csv_path)

    # -----------------------------
    # PLOT CONFUSION MATRIX
    # -----------------------------
    plt.figure(figsize=(8, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels
    )

    plt.title(f"{model_name} - {dataset_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    image_path = os.path.join(
        save_folder,
        f"{dataset_name}_{model_name}_confusion_matrix.png"
    )

    plt.savefig(image_path, bbox_inches='tight')
    plt.close()

    return results



In [5]:
#implementing models

svm = SVC(kernel='rbf', probability=True)
dt = DecisionTreeClassifier()
rf = RandomForestClassifier(n_estimators=100)
adaboost = AdaBoostClassifier(n_estimators=100)
nb = GaussianNB()
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
xgboost = XGBClassifier(use_label_encoder=False,eval_metric='logloss')
catboost = CatBoostClassifier(verbose=0)

In [6]:

# =========================================================
# CREATE FOLDER TO SAVE CONFUSION MATRICES
# =========================================================
cm_folder = "confusion_matrices"
os.makedirs(cm_folder, exist_ok=True)


# =========================================================
# MODELS
# =========================================================
models = {

    "SVM": SVC(
        kernel='rbf',
        probability=True,
        gamma='scale',
        C=1.0,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        criterion='gini',
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=6,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=18,
        min_samples_split=6,
        min_samples_leaf=3,
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=130,
        learning_rate=0.15,
        random_state=42
    ),

    "Naive Bayes": GaussianNB(
        var_smoothing=3.7e-8
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(100, 50),
        activation='relu',
        alpha=0.004,
        learning_rate_init=0.006,
        max_iter=1200,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.025,
        subsample=0.75,
        colsample_bytree=0.80,
        eval_metric='logloss',
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        iterations=150,
        depth=5,
        learning_rate=0.03,
        verbose=0,
        random_state=42
    ),
}
# =========================================================
# DATASETS
# =========================================================
datasets = [
    (
        X_train1, X_test1,
        X_train_scaled1, X_test_scaled1,
        y_train1, y_test1,
        "protbert"
    ),

    (
        X_train2, X_test2,
        X_train_scaled2, X_test_scaled2,
        y_train2, y_test2,
        "bert"
    ),

    (
        X_train3, X_test3,
        X_train_scaled3, X_test_scaled3,
        y_train3, y_test3,
        "roberta"
    ),
]


# =========================================================
# PROCESSING LOOP
# =========================================================
all_results = []

for X_tr, X_te, X_tr_sc, X_te_sc, y_tr, y_te, label in datasets:

    for name, model in models.items():

        print(f"\nRunning {name} on {label}...")

        # Scaling for SVM and MLP
        if name in ["SVM", "MLP"]:
            current_X_train = X_tr_sc
            current_X_test = X_te_sc
        else:
            current_X_train = X_tr
            current_X_test = X_te

        # Evaluate model
        res = evaluate_model(
            model=model,
            X_train=current_X_train,
            X_test=current_X_test,
            y_train=y_tr,
            y_test=y_te,
            dataset_name=label,
            model_name=name.replace(" ", "_"),
            save_folder=cm_folder
        )

        # Add metadata
        res["Model"] = name
        res["Dataset"] = label

        all_results.append(res)


# =========================================================
# FINAL RESULTS DATAFRAME
# =========================================================
results_df = pd.DataFrame(all_results)

column_order = [
    "Dataset",
    "Model",

    "Train Accuracy",
    "Test Accuracy",

    "Train Precision",
    "Test Precision",

    "Train Recall",
    "Test Recall",

    "Train F1",
    "Test F1"
]

results_df = results_df[column_order]

# Sort results
results_df = results_df.sort_values(
    by=["Dataset", "Test F1"],
    ascending=[True, False]
)

# =========================================================
# SAVE FINAL RESULTS
# =========================================================
results_df.to_csv("all_model_results.csv", index=False)

print("\n=================================================")
print("Training Complete")
print("=================================================")
print("Confusion matrices saved in:", cm_folder)
print("Results saved as: all_model_results.csv")

# Display results
print(results_df)


Running SVM on protbert...

Running Decision Tree on protbert...

Running Random Forest on protbert...

Running AdaBoost on protbert...

Running Naive Bayes on protbert...

Running MLP on protbert...

Running XGBoost on protbert...

Running CatBoost on protbert...

Running SVM on bert...

Running Decision Tree on bert...

Running Random Forest on bert...

Running AdaBoost on bert...

Running Naive Bayes on bert...

Running MLP on bert...

Running XGBoost on bert...

Running CatBoost on bert...

Running SVM on roberta...

Running Decision Tree on roberta...

Running Random Forest on roberta...

Running AdaBoost on roberta...

Running Naive Bayes on roberta...

Running MLP on roberta...

Running XGBoost on roberta...

Running CatBoost on roberta...

Training Complete
Confusion matrices saved in: confusion_matrices
Results saved as: all_model_results.csv
     Dataset          Model  Train Accuracy  Test Accuracy  Train Precision  \
10      bert  Random Forest        0.991247       0.9978

In [7]:
results_df[::-1]

,Dataset,Model,Train Accuracy,Test Accuracy,Train Precision,Test Precision,Train Recall,Test Recall,Train F1,Test F1
20,roberta,Naive Bayes,0.977024,0.986871,0.977820,0.987132,0.977024,0.986871,0.976848,0.986812
17,roberta,Decision Tree,0.987965,0.986871,0.987974,0.986949,0.987965,0.986871,0.987949,0.986889
21,roberta,MLP,0.988512,0.995624,0.988715,0.995653,0.988512,0.995624,0.988470,0.995617
19,roberta,AdaBoost,0.989059,0.995624,0.989243,0.995653,0.989059,0.995624,0.989021,0.995617
16,roberta,SVM,0.985777,0.995624,0.986086,0.995653,0.985777,0.995624,0.985712,0.995617
23,roberta,CatBoost,0.992888,0.997812,0.992967,0.997819,0.992888,0.997812,0.992873,0.997810
22,roberta,XGBoost,0.992888,0.997812,0.992967,0.997819,0.992888,0.997812,0.992873,0.997810
18,roberta,Random Forest,0.992341,0.997812,0.992432,0.997819,0.992341,0.997812,0.992323,0.997810
4,protbert,Naive Bayes,0.861538,0.892720,0.885316,0.905004,0.861538,0.892720,0.863462,0.894549
0,protbert,SVM,0.918269,0.915709,0.924682,0.921842,0.918269,0.915709,0.919061,0.916774
